In [1]:
import numpy as np
import math
from scipy.sparse import csr_matrix
from utils import generate_laplacian, get_circuit_unitary
from foqcs_lcu.lcu import lcu_block_encoding

from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliGate

from qiskit import transpile
from qiskit_aer import AerSimulator

In [2]:
def PauliDecomposition(matrix, sparse=False, PauliStringInit="", output="Lists", tolerance=1e-6):
    """
        Computes the Pauli decomposition of a square matrix.

        Iteratively splits tensor factors off and decomposes those smaller
        matrices. This is done using submatrices of the original matrix.
        The Pauli strings are generated in each step.

        Args:
            matrix: Matrix to be decomposed
                    (Preferably numpy array/scipysparse).
            output: How the output should be generated.
            sparse: Whether matrix is in sparse format.
            PauliStringInit: For recursive computation.
            tolerance: Threshold below which coefficients are ignored.

        Returns:
            decomposition/outString: String of 1XYZ with their factors.
    """
    matDim = matrix.shape[0]
    qBitDim = math.ceil(np.log(matDim) / np.log(2))

    # Pad, if dimension is not a power of 2
    padDim = 2**qBitDim - matDim
    if padDim != 0:  # This condition states that the padding happens only if needed
        if sparse:
            indxptr = np.pad(matrix.indptr, ((0, padDim), (0, padDim)))
            matrix = csr_matrix((matrix.data, matrix.indices, indxptr))
        else:
            matrix = np.pad(matrix, ((0, padDim), (0, padDim)))

    decomposition = []

    if output == "Lists":
        Strings = []
        Coeffs = []

    # Output for dimension 1
    if qBitDim == 0:
        if abs(matrix[0, 0]) > tolerance:  # Ignore small coefficients
            if output == "Lists":
                Strings.append(PauliStringInit)
                try:
                    Coeffs.append(matrix[0, 0])
                except:
                    Coeffs.append(matrix[0, 0].numpy())
            else:
                decomposition = [f"{matrix[0, 0]}, {PauliStringInit}. "]

    # Calculates the tensor product coefficients via the sliced submatrices.
    if qBitDim > 0:
        halfDim = int(2**(qBitDim - 1))

        coeff1 = 0.5 * (matrix[0:halfDim, 0:halfDim] + matrix[halfDim:, halfDim:])
        coeffX = 0.5 * (matrix[halfDim:, 0:halfDim] + matrix[0:halfDim, halfDim:])
        coeffY = -1.j * 0.5 * (matrix[halfDim:, 0:halfDim] - matrix[0:halfDim, halfDim:])
        coeffZ = 0.5 * (matrix[0:halfDim, 0:halfDim] - matrix[halfDim:, halfDim:])

        coefficients = {"I": coeff1, "X": coeffX, "Y": coeffY, "Z": coeffZ}

        matrix = None

        # Recursion for the submatrices
        for c in coefficients:
            mat = coefficients[c]
            if sparse:
                nonZero = len(mat.nonzero()[0])
            else:
                nonZero = np.any(np.abs(mat) > tolerance)  # Check for significant values
            
            # Ignore zero or near-zero matrices
            if nonZero:
                subDec = PauliDecomposition(
                    mat, sparse, f"{PauliStringInit}{c}", output, tolerance
                )
                if output == "Lists":
                    Strings.extend(subDec[0])
                    Coeffs.extend(subDec[1])
                else:
                    decomposition.append(subDec)

    if output == "Lists":
        return [Strings, Coeffs]
    else:
        outputString = "".join(decomposition)
        return outputString

# Test for a Dirichlet Laplacian

Matrix size 4x4

In [3]:
ops, coefs = PauliDecomposition(generate_laplacian([4], analytic_normalize=True).todense())
ops2 = []

for op in ops:
    ops2.append(PauliGate(op))
    
be = lcu_block_encoding(2, np.sqrt(coefs), ops2)

In [4]:
qc = QuantumCircuit(4)
qc.append(be, [0, 1, 2, 3])
qc.draw()

┌──────┐
q_0: ┤0     ├
     │      │
q_1: ┤1     ├
     │  LCU │
q_2: ┤2     ├
     │      │
q_3: ┤3     ├
     └──────┘

In [ ]:
get_circuit_unitary(qc, [2])

Function call - get_circuit_unitary
Simulator loaded
Starting transpilation
